# Problema 2: zona habitable de estrellas de baja masa (0.1 a 1.2 Msun)

**Objetivo:** graficar la zona habitable (ZH) de estrellas de hasta 1.2 Msun usando un modelo de evolucion estelar (L y Teff en funcion de la masa) y las ecuaciones de Kopparapu et al. (2013).

**Flujo:** tabla del modelo estelar → ecuaciones de la ZH → prueba con el Sol → L(M) y Teff(M) → bordes de la ZH para cada masa → graficas → aplicacion a LHS 1140.

**Modelo estelar:** trayectorias de Baraffe et al. (2015, A&A 577, A42; 0.07 a 1.4 Msun, composicion solar Z = 0.0153), evaluadas a varias edades con el modulo `stellar` de VPLanet (Barnes et al. 2020), que interpola esas trayectorias. La tabla ya viene incluida en el notebook, asi que no hace falta descargar nada.

**Criterio de validacion:** que el Sol salga cerca de lo esperado (Kopparapu et al. 2013: borde interno 0.99 ua y externo 1.70 ua en el caso conservador).

## Importaciones

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

### Constantes del sistema

Valores de LHS 1140 tomados de Cadieux et al. (2024). Es el mismo bloque en los notebooks de RV, tránsitos y zona habitable.

In [ ]:
# ============================================================================
# Constantes de LHS 1140 (Cadieux et al. 2024, arXiv:2310.15490)
# Mismo bloque en los notebooks de RV, tránsitos y zona habitable.
# Si cambias un valor aquí, cámbialo también en los otros dos.
# ============================================================================
P_ORB   = {"b": 24.73723, "c": 3.777940}        # d, periodo orbital (Tabla D1)
T0_BTJD = {"b": 1399.9300, "c": 1389.2939}      # BJD - 2457000, tiempo de tránsito (Tabla D1)
M_STAR, M_STAR_ERR = 0.1844, 0.0045             # M_sol (Tabla C1)
R_STAR, R_STAR_ERR = 0.2159, 0.0030             # R_sol (Tabla C1)
TEFF_STAR = 3096.0                              # K (Tabla C1)
P_ROT = 131.0                                   # d, periodo de rotación (Tabla C1)


## Tabla del modelo estelar

Para cada edad (Gyr): masa M (Msun), luminosidad L (Lsun), temperatura efectiva Teff (K) y radio R (Rsun).

Las trayectorias de Baraffe et al. (2015) terminan cuando la estrella sale de la secuencia principal: para 1.15 y 1.2 Msun no hay modelo a 4.6 Gyr (la trayectoria del modelo de 1.2 Msun termina entre 4 y 4.5 Gyr). Por eso la edad de referencia de las graficas es 4 Gyr, la mayor para la que existen todas las masas hasta 1.2 Msun. Se omite 0.08 Msun porque su Teff (~2400 K) queda fuera del rango de validez de Kopparapu.

In [ ]:
# tabla del modelo: Baraffe et al. (2015) evaluada con VPLanet
modelo = {
    1.0: {
        "M": [0.1, 0.125, 0.15, 0.175, 0.1844, 0.2, 0.225, 0.25, 0.275, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1, 1.05, 1.1, 1.15, 1.2],
        "L": [0.000868788, 0.00167818, 0.00262422, 0.00359838, 0.00390347, 0.00477529, 0.00563448, 0.00689885, 0.00852239, 0.0103276, 0.0141868, 0.0190797, 0.0256823, 0.0348337, 0.048573, 0.0683912, 0.0963584, 0.134928, 0.185784, 0.252173, 0.336607, 0.443672, 0.579489, 0.749501, 0.960454, 1.21924, 1.53378, 1.91012],
        "Teff": [2810.0, 3014.9, 3130.0, 3201.2, 3219.2, 3261.0, 3294.4, 3335.3, 3377.7, 3415.0, 3467.5, 3520.0, 3589.0, 3680.0, 3807.5, 3967.0, 4166.0, 4386.1, 4607.0, 4825.7, 5027.3, 5219.9, 5410.2, 5591.8, 5759.8, 5915.1, 6055.7, 6186.9],
        "R": [0.1240, 0.1507, 0.1740, 0.1952, 0.2006, 0.2170, 0.2319, 0.2505, 0.2709, 0.2910, 0.3306, 0.3720, 0.4152, 0.4600, 0.5073, 0.5540, 0.5963, 0.6370, 0.6771, 0.7190, 0.7659, 0.8159, 0.8674, 0.9230, 0.9848, 1.0522, 1.1263, 1.2040],
    },
    2.0: {
        "M": [0.1, 0.125, 0.15, 0.175, 0.1844, 0.2, 0.225, 0.25, 0.275, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1, 1.05, 1.1, 1.15, 1.2],
        "L": [0.000866962, 0.00168162, 0.00262422, 0.0035989, 0.00390508, 0.0047863, 0.00566164, 0.00695582, 0.00862115, 0.0104698, 0.0143826, 0.0193145, 0.0259869, 0.0352371, 0.0491019, 0.069173, 0.0978475, 0.137713, 0.190377, 0.259655, 0.348999, 0.464032, 0.612782, 0.802284, 1.0409, 1.33841, 1.70678, 2.15344],
        "Teff": [2810.0, 3014.9, 3130.0, 3201.2, 3219.2, 3261.0, 3294.4, 3335.4, 3377.7, 3415.0, 3467.0, 3519.0, 3587.8, 3679.0, 3807.5, 3968.9, 4171.5, 4395.8, 4619.9, 4841.9, 5047.0, 5243.6, 5439.9, 5626.3, 5793.5, 5944.6, 6080.7, 6202.0],
        "R": [0.1240, 0.1507, 0.1740, 0.1952, 0.2005, 0.2170, 0.2323, 0.2515, 0.2725, 0.2930, 0.3328, 0.3740, 0.4171, 0.4620, 0.5098, 0.5570, 0.5991, 0.6399, 0.6813, 0.7250, 0.7738, 0.8266, 0.8820, 0.9434, 1.0139, 1.0924, 1.1791, 1.2727],
    },
    3.0: {
        "M": [0.1, 0.125, 0.15, 0.175, 0.1844, 0.2, 0.225, 0.25, 0.275, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1, 1.05, 1.1, 1.15, 1.2],
        "L": [0.000866962, 0.00168134, 0.00263039, 0.00360661, 0.00391084, 0.00479733, 0.00569186, 0.00702422, 0.0087423, 0.0106398, 0.0145603, 0.0194454, 0.0261423, 0.0354813, 0.049535, 0.0699732, 0.0992827, 0.140272, 0.194846, 0.267397, 0.362344, 0.486579, 0.649991, 0.862746, 1.13963, 1.49161, 1.93261, 2.45716],
        "Teff": [2810.0, 3015.7, 3131.0, 3201.1, 3219.1, 3261.0, 3294.6, 3335.9, 3378.5, 3416.0, 3468.1, 3520.0, 3588.0, 3679.0, 3809.6, 3974.0, 4179.3, 4406.5, 4633.6, 4859.0, 5068.5, 5269.7, 5471.3, 5661.1, 5828.0, 5973.2, 6103.1, 6192.7],
        "R": [0.1240, 0.1507, 0.1740, 0.1951, 0.2004, 0.2170, 0.2327, 0.2526, 0.2741, 0.2950, 0.3343, 0.3750, 0.4186, 0.4640, 0.5118, 0.5590, 0.6015, 0.6430, 0.6855, 0.7308, 0.7818, 0.8378, 0.8977, 0.9661, 1.0479, 1.1415, 1.2443, 1.3633],
    },
    4.0: {
        "M": [0.1, 0.125, 0.15, 0.175, 0.1844, 0.2, 0.225, 0.25, 0.275, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1, 1.05, 1.1, 1.15, 1.2],
        "L": [0.000866962, 0.00168222, 0.00263027, 0.0036071, 0.00391222, 0.00480839, 0.00572156, 0.00708937, 0.0088558, 0.0107975, 0.0147177, 0.0195511, 0.0262805, 0.0357273, 0.0499854, 0.0708067, 0.100756, 0.142891, 0.199525, 0.275648, 0.376475, 0.510771, 0.691987, 0.934216, 1.26487, 1.68333, 2.12453, 2.68136],
        "Teff": [2811.0, 3015.9, 3131.0, 3202.2, 3220.2, 3262.0, 3295.4, 3336.4, 3378.7, 3416.0, 3468.0, 3520.0, 3588.3, 3680.0, 3812.2, 3979.0, 4187.5, 4418.3, 4648.6, 4877.4, 5091.0, 5296.6, 5503.7, 5696.5, 5866.9, 5995.4, 6026.0, 6056.8],
        "R": [0.1240, 0.1507, 0.1740, 0.1951, 0.2003, 0.2170, 0.2332, 0.2536, 0.2758, 0.2970, 0.3359, 0.3760, 0.4194, 0.4650, 0.5132, 0.5610, 0.6039, 0.6460, 0.6892, 0.7360, 0.7896, 0.8499, 0.9157, 0.9934, 1.0898, 1.2039, 1.3463, 1.4888],
    },
    4.6: {
        "M": [0.1, 0.125, 0.15, 0.175, 0.1844, 0.2, 0.225, 0.25, 0.275, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1, 1.05, 1.1],
        "L": [0.000866962, 0.00168222, 0.00263027, 0.00361571, 0.0039208, 0.00481948, 0.00574018, 0.00712239, 0.0089088, 0.0108699, 0.0148018, 0.0196336, 0.0263885, 0.0358909, 0.0502555, 0.0712857, 0.101641, 0.144523, 0.202432, 0.280804, 0.385518, 0.526739, 0.721046, 0.984705, 1.33624, 1.81328],
        "Teff": [2811.0, 3015.9, 3131.0, 3202.2, 3220.2, 3262.0, 3295.4, 3336.3, 3378.7, 3416.0, 3468.5, 3521.0, 3589.3, 3681.0, 3814.0, 3982.0, 4192.1, 4424.9, 4657.3, 4888.4, 5104.9, 5313.5, 5524.1, 5717.3, 5854.5, 5995.1],
        "R": [0.1240, 0.1507, 0.1740, 0.1952, 0.2007, 0.2180, 0.2342, 0.2547, 0.2768, 0.2980, 0.3369, 0.3770, 0.4204, 0.4660, 0.5142, 0.5620, 0.6047, 0.6470, 0.6914, 0.7400, 0.7949, 0.8576, 0.9275, 1.0124, 1.1309, 1.2493],
    },
}

## Ecuaciones de Kopparapu et al. (2013)

El flujo estelar efectivo en cada borde de la ZH es

$$
S_{eff} = S_{eff,\odot} + aT_\star + bT_\star^2 + cT_\star^3 + dT_\star^4, \qquad T_\star = T_{eff} - 5780\ \mathrm{K}
$$

y la distancia del borde a la estrella (en ua) es

$$
d = \sqrt{\frac{L/L_\odot}{S_{eff}}}.
$$

Se usan los coeficientes de la Tabla 3 (planeta de 1 masa terrestre), validos para 2600 K <= Teff <= 7200 K.
Cada borde corresponde a un limite climatico: *Recent Venus* (interno optimista), *Moist Greenhouse* (interno conservador), *Maximum Greenhouse* (externo conservador) y *Early Mars* (externo optimista).

In [ ]:
KOP = {  # kopparapu et al. (2013), Tabla 3: Seff_sun, a, b, c, d
    "Recent Venus": (1.7753, 1.4316e-4, 2.9875e-9, -7.5702e-12, -1.1635e-15),
    "Moist Greenhouse": (1.0140, 8.1774e-5, 1.7063e-9, -4.3241e-12, -6.6462e-16),
    "Maximum Greenhouse": (0.3438, 5.8942e-5, 1.6558e-9, -3.0045e-12, -5.2983e-16),
    "Early Mars": (0.3179, 5.4513e-5, 1.5313e-9, -2.7786e-12, -4.8997e-16),
}

def seff(nombre, teff):
    s0, a, b, c, d = KOP[nombre]
    T = teff - 5780.0
    return s0 + a * T + b * T**2 + c * T**3 + d * T**4

def borde_hz(nombre, teff, lum):
    return np.sqrt(lum / seff(nombre, teff))  # ua

## Prueba con el Sol

Se calculan los bordes con el Sol real (Teff = 5772 K, L = 1) y con el modelo de 1.0 Msun a 4.6 Gyr. Tambien se mira donde caen Venus, la Tierra y Marte.

In [ ]:
orden = ["Recent Venus", "Moist Greenhouse", "Maximum Greenhouse", "Early Mars"]
i_sol = modelo[4.6]["M"].index(1.0)
L_sol_mod = modelo[4.6]["L"][i_sol]
T_sol_mod = modelo[4.6]["Teff"][i_sol]

casos = [("Sol real (5772 K, L = 1)", 5772.0, 1.0),
         ("Modelo 1.0 Msun a 4.6 Gyr", T_sol_mod, L_sol_mod)]
for etiqueta, teff, lum in casos:
    b = [borde_hz(n, teff, lum) for n in orden]
    print(f"{etiqueta}: L = {lum:.4f} Lsun, Teff = {teff:.0f} K")
    print(f"   conservadora: {b[1]:.3f} a {b[2]:.3f} ua | optimista: {b[0]:.3f} a {b[3]:.3f} ua")

# referencia: Kopparapu et al. (2013), Sol: 0.99 y 1.70 ua (conservadora)
b_real = [borde_hz(n, 5772.0, 1.0) for n in orden]
print("Sol real vs literatura: interno", round(b_real[1], 3), "(0.99) | externo", round(b_real[2], 3), "(1.70)")

planetas_sol = [("Venus", 0.723), ("Tierra", 1.000), ("Marte", 1.524)]
b_mod = [borde_hz(n, T_sol_mod, L_sol_mod) for n in orden]
for nombre, dist in planetas_sol:
    en_con = b_mod[1] <= dist <= b_mod[2]
    en_opt = b_mod[0] <= dist <= b_mod[3]
    print(f"{nombre} ({dist} ua): dentro de la ZH conservadora = {en_con} | optimista = {en_opt}")

## L, Teff y R en funcion de la masa

Esto es el modelo de evolucion estelar: cada curva es una edad. Se marcan el Sol y LHS 1140 con sus valores medidos (Teff y R de Cadieux et al. 2024; L calculada de ambos).

In [ ]:
L_OBS = R_STAR**2 * (TEFF_STAR / 5772.0)**4

fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
colores = plt.cm.viridis(np.linspace(0, 0.9, len(modelo)))
for color, edad in zip(colores, modelo):
    m = modelo[edad]
    ax[0].plot(m["M"], m["L"], color=color, label=f"{edad} Gyr")
    ax[1].plot(m["M"], m["Teff"], color=color)
    ax[2].plot(m["M"], m["R"], color=color)
ax[0].scatter([1.0, M_STAR], [1.0, L_OBS], color=["k", "tab:red"], zorder=3)
ax[1].scatter([1.0, M_STAR], [5772.0, TEFF_STAR], color=["k", "tab:red"], zorder=3)
ax[2].scatter([1.0, M_STAR], [1.0, R_STAR], color=["k", "tab:red"], zorder=3)
ax[0].set_yscale("log")
ax[0].set_ylabel("L (Lsun)"); ax[1].set_ylabel("Teff (K)"); ax[2].set_ylabel("R (Rsun)")
for a in ax:
    a.set_xlabel("masa (Msun)")
ax[0].legend(title="edad", fontsize=8)
ax[0].annotate("Sol", (1.0, 1.0), xytext=(-25, 8), textcoords="offset points", fontsize=8)
ax[0].annotate("LHS 1140", (M_STAR, L_OBS), xytext=(8, -12), textcoords="offset points", fontsize=8, color="tab:red")
plt.tight_layout()
plt.show()

## Bordes de la zona habitable para cada masa

Para cada edad y cada masa de la tabla se calculan los cuatro bordes. Se comprueba que todas las Teff esten dentro del rango de validez de Kopparapu (2600 a 7200 K).

In [ ]:
hz = {}
for edad in modelo:
    m = modelo[edad]
    h = {"M": m["M"], "in_opt": [], "in_con": [], "out_con": [], "out_opt": []}
    for teff, lum in zip(m["Teff"], m["L"]):
        h["in_opt"].append(borde_hz("Recent Venus", teff, lum))
        h["in_con"].append(borde_hz("Moist Greenhouse", teff, lum))
        h["out_con"].append(borde_hz("Maximum Greenhouse", teff, lum))
        h["out_opt"].append(borde_hz("Early Mars", teff, lum))
    hz[edad] = h

fuera = 0
for edad in modelo:
    for teff in modelo[edad]["Teff"]:
        if teff < 2600.0 or teff > 7200.0:
            fuera += 1
print("Modelos fuera del rango de Kopparapu (2600 a 7200 K):", fuera)

## Grafica de la zona habitable

Izquierda: ZH en funcion de la masa a 4 Gyr (banda oscura = conservadora, clara = optimista), con Venus, la Tierra y Marte, y los planetas b y c de LHS 1140. Derecha: como se desplazan los bordes conservadores con la edad.

In [ ]:
edad_ref = 4.0
h = hz[edad_ref]
planetas_lhs = [("b", 0.0946), ("c", 0.0270)]   # semiejes de la seccion de transitos (ua)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
a = ax[0]
a.fill_between(h["M"], h["in_opt"], h["out_opt"], color="tab:green", alpha=0.2, label="ZH optimista")
a.fill_between(h["M"], h["in_con"], h["out_con"], color="tab:green", alpha=0.45, label="ZH conservadora")
for nombre, dist in planetas_sol:
    a.scatter([1.0], [dist], color="k", s=25, zorder=3)
    a.annotate(nombre, (1.0, dist), xytext=(6, 0), textcoords="offset points", fontsize=8, va="center")
for nombre, dist in planetas_lhs:
    a.scatter([M_STAR], [dist], color="tab:red", s=35, zorder=3)
    a.annotate(f"LHS 1140 {nombre}", (M_STAR, dist), xytext=(6, 0), textcoords="offset points", fontsize=8, va="center", color="tab:red")
a.set_yscale("log")
a.set_xlabel("masa de la estrella (Msun)"); a.set_ylabel("distancia (ua)")
a.set_title(f"Zona habitable a {edad_ref} Gyr")
a.legend(loc="upper left", fontsize=8)

a = ax[1]
colores = plt.cm.viridis(np.linspace(0, 0.9, len(hz)))
for color, edad in zip(colores, hz):
    a.plot(hz[edad]["M"], hz[edad]["in_con"], color=color, label=f"{edad} Gyr")
    a.plot(hz[edad]["M"], hz[edad]["out_con"], color=color)
a.set_yscale("log")
a.set_xlabel("masa de la estrella (Msun)"); a.set_ylabel("distancia (ua)")
a.set_title("Bordes conservadores segun la edad")
a.legend(title="edad", fontsize=8)
plt.tight_layout()
plt.show()

## Zona habitable general en malla fina de masa

El modelo tabulado tiene 28 masas por edad. Aqui se interpola $\log L$ y $T_\mathrm{eff}$ en $\log M$ para obtener una malla de $0.01\ M_\odot$ entre 0.1 y 1.2 $M_\odot$ y se recalculan los cuatro bordes de Kopparapu. No se extrapola: donde una edad no tiene modelo (1.15 y 1.2 $M_\odot$ a 4.6 Gyr) queda NaN. Paneles: ZH frente a la masa (4 Gyr), cambio de los bordes con la edad y ZH expresada en periodo orbital. Genera `fig_zh_masa.png`.

In [ ]:
# ---- Celda nueva: ZH general en malla fina de masa (interpolando el modelo) ----
M_fino = np.round(np.arange(0.10, 1.2001, 0.01), 2)

def interpola(edad, M_new):
    m = modelo[edad]
    lm = np.log10(m["M"])
    L = 10 ** np.interp(np.log10(M_new), lm, np.log10(m["L"]))
    T = np.interp(np.log10(M_new), lm, m["Teff"])
    # no extrapolar: fuera de la tabla de esa edad queda NaN
    fuera = (M_new < min(m["M"])) | (M_new > max(m["M"]))
    L[fuera] = np.nan; T[fuera] = np.nan
    return L, T

hz_f = {}
for edad in modelo:
    L, T = interpola(edad, M_fino)
    hz_f[edad] = {"L": L, "Teff": T,
                  "in_opt": borde_hz("Recent Venus", T, L),
                  "in_con": borde_hz("Moist Greenhouse", T, L),
                  "out_con": borde_hz("Maximum Greenhouse", T, L),
                  "out_opt": borde_hz("Early Mars", T, L)}

def periodo(a, M):          # d, con a en ua y M en Msun (Kepler)
    return 365.25 * np.sqrt(a**3 / M)

edad_ref = 4.0
h = hz_f[edad_ref]
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))

# (1) ZH vs masa, 4 Gyr
a = ax[0]
a.fill_between(M_fino, h["in_opt"], h["out_opt"], color="tab:green", alpha=0.2, label="ZH optimista")
a.fill_between(M_fino, h["in_con"], h["out_con"], color="tab:green", alpha=0.45, label="ZH conservadora")
for nombre, dist in planetas_sol:
    a.scatter([1.0], [dist], color="k", s=25, zorder=3)
    a.annotate(nombre, (1.0, dist), xytext=(6, 0), textcoords="offset points", fontsize=8, va="center")
for nombre, dist in planetas_lhs:
    a.scatter([M_STAR], [dist], color="tab:red", s=35, zorder=3)
    a.annotate(f"b" if nombre == "b" else "c", (M_STAR, dist), xytext=(6, 0), textcoords="offset points", fontsize=8, color="tab:red", va="center")
a.set_yscale("log"); a.set_xlabel("masa de la estrella (Msun)"); a.set_ylabel("distancia (ua)")
a.set_title(f"Zona habitable a {edad_ref} Gyr"); a.legend(loc="upper left", fontsize=8)

# (2) cambio de los bordes con la edad, respecto a 1 Gyr
a = ax[1]
for M0, c in zip([0.2, 0.5, 0.8, 1.0, 1.1, 1.2], plt.cm.plasma(np.linspace(0, 0.85, 6))):
    k = int(np.argmin(abs(M_fino - M0)))
    edades = [e for e in hz_f if not np.isnan(hz_f[e]["in_con"][k])]
    a.plot(edades, [100 * (hz_f[e]["out_con"][k] / hz_f[1.0]["out_con"][k] - 1) for e in edades], "o-", color=c, label=f"{M0} Msun")
a.set_xlabel("edad (Gyr)"); a.set_ylabel("cambio del borde externo respecto a 1 Gyr (%)")
a.set_title("Dependencia con la edad"); a.legend(fontsize=8)

# (3) la misma ZH en periodo orbital
a = ax[2]
a.fill_between(M_fino, periodo(h["in_opt"], M_fino), periodo(h["out_opt"], M_fino), color="tab:green", alpha=0.2)
a.fill_between(M_fino, periodo(h["in_con"], M_fino), periodo(h["out_con"], M_fino), color="tab:green", alpha=0.45)
for nombre, P in P_ORB.items():
    a.scatter([M_STAR], [P], color="tab:red", s=35, zorder=3); a.annotate(nombre, (M_STAR, P), xytext=(6, 0), textcoords="offset points", fontsize=8, color="tab:red", va="center")
a.scatter([1.0], [365.25], color="k", s=25, zorder=3); a.annotate("Tierra", (1.0, 365.25), xytext=(6, 0), textcoords="offset points", fontsize=8, va="center")
a.set_yscale("log"); a.set_xlabel("masa de la estrella (Msun)"); a.set_ylabel("periodo orbital (d)")
a.set_title("ZH en periodo orbital (4 Gyr)")
plt.tight_layout(); plt.savefig("fig_zh_masa.png", dpi=200); plt.show()


### Periodos orbitales, leyes de potencia y dependencia con la edad

In [ ]:
# Tabla con periodos orbitales en los bordes conservadores (4 Gyr)
print(f"{'M (Msun)':>9} {'L (Lsun)':>10} {'Teff (K)':>9} {'a_int (ua)':>11} {'a_ext (ua)':>11} {'P_int (d)':>10} {'P_ext (d)':>10}")
for masa in [0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.1, 1.2]:
    k = int(np.argmin(abs(M_fino - masa)))
    ai, ao = h["in_con"][k], h["out_con"][k]
    print(f"{masa:9.2f} {h['L'][k]:10.5f} {h['Teff'][k]:9.0f} {ai:11.4f} {ao:11.4f} {periodo(ai, masa):10.1f} {periodo(ao, masa):10.1f}")

# Leyes de potencia (4 Gyr): L ~ M^n y borde interno ~ M^p
for lo, hi in [(0.1, 0.5), (0.5, 1.0)]:
    s = (M_fino >= lo) & (M_fino <= hi)
    n = np.polyfit(np.log10(M_fino[s]), np.log10(h["L"][s]), 1)[0]
    p = np.polyfit(np.log10(M_fino[s]), np.log10(h["in_con"][s]), 1)[0]
    print(f"{lo}-{hi} Msun: L ~ M^{n:.2f}, borde interno ~ M^{p:.2f}")

# Cambio del borde externo conservador entre 1 y 4 Gyr
for M0 in [0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.2]:
    k = int(np.argmin(abs(M_fino - M0)))
    print(f"{M0:4.1f} Msun: {100 * (hz_f[4.0]['out_con'][k] / hz_f[1.0]['out_con'][k] - 1):+5.1f} %")

## Resumen por masa

Luminosidad, temperatura y bordes conservadores a 4 Gyr para algunas masas.

In [ ]:
print(f"{'M (Msun)':>9} {'L (Lsun)':>10} {'Teff (K)':>9} {'ZH interna (ua)':>16} {'ZH externa (ua)':>16}")
h = hz[4.0]
m = modelo[4.0]
for masa in [0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.1, 1.2]:
    i = m["M"].index(masa)
    print(f"{masa:9.2f} {m['L'][i]:10.5f} {m['Teff'][i]:9.0f} {h['in_con'][i]:16.4f} {h['out_con'][i]:16.4f}")

## Zona habitable de LHS 1140

La masa de LHS 1140 (0.1844 Msun) esta incluida en la tabla, asi que los valores del modelo salen directos (la interpolacion en log L y log M queda por si se usa otra masa). Se comparan con los valores medidos.

In [ ]:
edad = 4.6
lm = np.log10(modelo[edad]["M"])
L_mod = 10 ** np.interp(np.log10(M_STAR), lm, np.log10(modelo[edad]["L"]))
T_mod = np.interp(np.log10(M_STAR), lm, modelo[edad]["Teff"])
R_mod = np.interp(np.log10(M_STAR), lm, modelo[edad]["R"])

print(f"Modelo ({M_STAR} Msun, {edad} Gyr): L = {L_mod:.5f} Lsun, Teff = {T_mod:.0f} K, R = {R_mod:.4f} Rsun")
print(f"Medido (Cadieux 2024):       L = {L_OBS:.5f} Lsun, Teff = {TEFF_STAR:.0f} K, R = {R_STAR:.4f} Rsun")
print(f"Diferencia modelo - medido:  L {100 * (L_mod / L_OBS - 1):+.1f} %, Teff {T_mod - TEFF_STAR:+.0f} K, R {100 * (R_mod / R_STAR - 1):+.1f} %")
print()

for etiqueta, teff, lum in [("con el modelo", T_mod, L_mod), ("con L y Teff medidos", TEFF_STAR, L_OBS)]:
    b = [borde_hz(n, teff, lum) for n in orden]
    print(f"ZH {etiqueta}: conservadora {b[1]:.4f} a {b[2]:.4f} ua | optimista {b[0]:.4f} a {b[3]:.4f} ua")

print()
b = [borde_hz(n, TEFF_STAR, L_OBS) for n in orden]
for nombre, dist in planetas_lhs:
    flujo = L_OBS / dist**2
    en_con = b[1] <= dist <= b[2]
    en_opt = b[0] <= dist <= b[3]
    print(f"LHS 1140 {nombre}: a = {dist} ua, flujo = {flujo:.2f} S_tierra | ZH conservadora = {en_con} | optimista = {en_opt}")

## Limitaciones

1. **Tipo de planeta:** los coeficientes de la Tabla 3 son para un planeta de 1 masa terrestre con atmosfera tipo Tierra (H2O y CO2); LHS 1140 b tiene unas 5 a 7 masas terrestres, y para masas distintas Kopparapu et al. (2013) dan otras tablas.
2. **Sin nubes ni rotacion sincronica:** el modelo climatico es 1D y no incluye nubes. Alrededor de enanas M los planetas suelen estar en rotacion sincronica, lo que puede mover los bordes de la ZH.
3. **Modelo estelar:** Baraffe et al. (2015) con composicion solar (Z = 0.0153). Los modelos de baja masa tienden a dar radios algo menores que los medidos (en LHS 1140, ver la comparacion de arriba), pero L, que es lo que entra en la ZH, es mucho menos sensible.
4. **Edad:** para M >= 1.1 Msun la ZH cambia mucho con la edad; los modelos de 1.15 y 1.2 Msun no existen a 4.6 Gyr porque la trayectoria termina antes.
5. **Rango de validez:** Kopparapu et al. (2013) valen entre 2600 y 7200 K; por eso se omite 0.08 Msun (Teff ~ 2400 K).

## Referencias

- Baraffe, I., Homeier, D., Allard, F. & Chabrier, G. (2015), *New evolutionary models for pre-main sequence and main sequence low-mass stars down to the hydrogen-burning limit*, A&A, 577, A42.
- Barnes, R. et al. (2020), *The Virtual Planet Simulator: VPLanet*, PASP, 132, 024502.
- Kopparapu, R. K. et al. (2013), *Habitable Zones around Main-sequence Stars: New Estimates*, ApJ, 765, 131.
- Cadieux, C. et al. (2024), *New Mass and Radius Constraints on the LHS 1140 Planets*, arXiv:2310.15490.